# Эллиптические уравнения
## Выделение особенности

$$u(x, y) = v(x, y) + w(x, y)$$

$$\Delta w = 0$$

$$w(x, y) = \frac{2}{\pi} \arctan\left(\frac{y}{x}\right) - \frac{1}{\pi} \arctan\left(\frac{y - 0.5}{x}\right) - 0.5$$

 Проверим поведение на границе $x \to 0$:
 * Если $y > 0.5$: $\frac{2}{\pi} \arctan(+\infty) - \frac{1}{\pi}\arctan(+\infty) - 0.5 = 1 - 0.5 - 0.5 = 0$. ($u(0,y)=0$).
 
 * Если $0 < y < 0.5$: $\arctan(+\infty) - \arctan(-\infty) - 0.5 = 1 - (-0.5) - 0.5 = 1$. ($u(0,y)=1$).

 Проверим поведение в углу $(0,0)$, двигаясь по нижней границе $y = 0$ ($x > 0$):

 $$w(x, 0) = \frac{2}{\pi}(0) - \frac{1}{\pi} \arctan\left(\frac{-0.5}{x}\right) - 0.5 = \frac{1}{\pi} \arctan\left(\frac{0.5}{x}\right) - 0.5$$

 При стремлении в угол $(x \to 0)$, получаем $\frac{1}{\pi}\left(\frac{\pi}{2}\right) - 0.5 = 0$. 
 
 Угловой скачок устранен


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f_val = -2.0
L = 1


def get_u_W_matrix(N):
    h = L / N
# устойчивость схемы крест, 
# преимущество шазматной нумерации уузлов 
'''Хорошо распараллеливается. Можно обновить значчения во всех красных узлах одноверменно, 
т.к. они вообще не зависят друг от друга. ЬЛ же самое можно провернуть с черными узлами'''
    
    def w_sing(x, y):
        # Computing left border x = 0 to avoid 0 division
        if np.isclose(x, 0.0):
            if y < 0.5:
                return 1.0
            elif np.isclose(y, 0.5):
                return 0.5  # Averaging in discontinuity point
            else:
                return 0.0
        # Analytical function, modelingfunction jumps
        return (
            (2 / np.pi) * np.arctan(y / x)
            - (1 / np.pi) * np.arctan((y - 0.5) / x)
            - 0.5
        )

    # In advance compute values of apecialty in all grid nodes
    W_matrix_ = np.zeros((N + 1, N + 1))
    for m in range(N + 1):
        for l in range(N + 1):
            W_matrix_[m, l] = w_sing(m * h, l * h)

    # 2. Initialising grid for SMOOTH funciton v
    v = np.zeros((N + 1, N + 1))

    # 3. Sedtting new border condtitions for v (v = u_exact - w_sing)
    for m in range(N + 1):
        x = m * h
        # Low border y = 0
        v[m, 0] = 0.0 - W_matrix_[m, 0]
        # Up border y = 1
        v[m, N] = 0.0 - W_matrix_[m, N]

    for l in range(N + 1):
        y = l * h
        # left border x = 0
        if y < 0.5:
            u_left = 1.0
        elif np.isclose(y, 0.5):
            u_left = 0.5
        else:
            u_left = 0.0
        v[0, l] = u_left - W_matrix_[0, l]  # Here v[0, l] must be 0.0

        # Right border x = 1
        v[N, l] = 0.0 - W_matrix_[N, l]

    return v, W_matrix_


def seidel(N, epsilon):
    # Grid step
    h = L / N
    h2 = h**2

    # RHS f(x, y) = -2
    #  -h^2 * f = 2 * h^2
    f_term = -h2 * f_val

    # Initialize the solution grid
    u, W_matrix = get_u_W_matrix(N)

    iters = 0
    while True:
        # Saving copuy from previos iteration for checking ocnvergence
        u_old = u.copy()

        # 1st semi step: ipdating Red nodes (sum of indexes is even
        for m in range(1, N):
            for l in range(1, N):
                if (m + l) % 2 == 0:
                    u[m, l] = 0.25 * (
                        u[m - 1, l] + u[m + 1, l] + u[m, l - 1] + u[m, l + 1] + f_term
                    )

        # 2nd semistep: updating Black nodes (sum of indexes is odd)
        for m in range(1, N):
            for l in range(1, N):
                if (m + l) % 2 != 0:
                    # Here are used updated on 1st semistep Red neighbores
                    u[m, l] = 0.25 * (
                        u[m - 1, l] + u[m + 1, l] + u[m, l - 1] + u[m, l + 1] + f_term
                    )

        iters += 1

        # Checking convergence by Chebishevs norm (max divergence)
        diff = np.max(np.abs(u - u_old))
        if diff < epsilon:
            break
    u += W_matrix

    return u, iters


def ptim(N, epsilon):
    # Grid step
    h = L / N

    # Initial eqality: Delta u = -2  =>  -Delta u = 2
    # Operator A approximates -Delta, so f_val = 2

    # Counting ПТИМ parameters by theory
    delta = (8.0 / h**2) * np.sin(np.pi * h / 2) ** 2
    Delta = 8.0 / h**2

    gamma1 = delta / (2 * (1 + np.sqrt(delta / Delta)))
    gamma2 = np.sqrt(delta * Delta) / 4.0

    omega = 2.0 / np.sqrt(delta * Delta)
    tau = 2.0 / (gamma1 + gamma2)

    # Coefficient for iterating process
    omega_h2 = omega / h**2
    coef = 1.0 + 2.0 * omega_h2

    u, W_matrix = get_u_W_matrix(N)

    iters = 0
    while True:
        # Step 1: counting r = A*u - f
        r = np.zeros((N + 1, N + 1))
        for m in range(1, N):
            for l in range(1, N):
                Au = (
                    4 * u[m, l] - u[m - 1, l] - u[m + 1, l] - u[m, l - 1] - u[m, l + 1]
                ) / h**2
                r[m, l] = Au - (-f_val)

        # Step 2: Solving system (E + omega R*) w_tilde = -tau * r (Reverse stepping)
        w_tilde = np.zeros((N + 1, N + 1))
        # Go from right up angle to the left down
        for m in range(N - 1, 0, -1):
            for l in range(N - 1, 0, -1):
                w_tilde[m, l] = (
                    -tau * r[m, l] + omega_h2 * (w_tilde[m + 1, l] + w_tilde[m, l + 1])
                ) / coef

        # Step 3: Solving system (E + omega R) w = w_tilde (Straight steppping)
        w = np.zeros((N + 1, N + 1))
        # Go from lift down to right up
        for m in range(1, N):
            for l in range(1, N):
                w[m, l] = (
                    w_tilde[m, l] + omega_h2 * (w[m - 1, l] + w[m, l - 1])
                ) / coef

        # Updating solution
        u += w
        iters += 1

        # Checking convergence
        if np.max(np.abs(w)) < epsilon:
            break
    u += W_matrix

    return u, iters

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import MultipleLocator

methods = {0: 'Seidel', 1: 'PTIM'}

def draw(N, epsilon, method):

    x = np.linspace(0, L, N + 1)  
    y = np.linspace(0, L, N + 1)  
    X, Y = np.meshgrid(x, y)  

    if method == 0:
        u_res, iter = seidel(N, epsilon)
    elif method == 1:
        u_res, iter = ptim(N, epsilon)
    Z = u_res.T

    fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(10, 7))

    surf = ax.plot_surface(X, Y, Z, cmap=cm.viridis, linewidth=0, antialiased=False)

    ax.xaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("u(x, y)")

    ax.set_title(f"Method: {methods[method]} | N: {N} | Iterations: {iter} | $\epsilon$: {epsilon}")

    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.show()


draw(100, 1e-6, 1)

## Зависимость числа итераций от размера сетки
е. $i=\frac{N^2}{\pi^2}\text{ln} \epsilon^{-1}$

к. $i=\frac{\sqrt{N}}{\sqrt{2\pi}}\text{ln} \epsilon^{-1}$

In [ ]:
method_form = {
    0: r"$i=\frac{N^2}{\pi^2}\text{ln} \epsilon^{-1}$",
    1: r"$i=\frac{\sqrt{N}}{\sqrt{2\pi}}\text{ln} \epsilon^{-1}$",
}


def draw_iters(N_values, epsilon, method):

    actual_iters = []
    theoretical_iters = []

    print(f"{'N':<5} | {'Iteratins (real)':<15} | {'Iterations (theory)':<15}")
    print("-" * 45)

    for N in N_values:

        if method == 0:
            u, it = seidel(N, epsilon)
            it_theory = (N**2 / np.pi**2) * np.log(1.0 / epsilon)
        elif method == 1:
            u, it = ptim(N, epsilon)
            it_theory = (np.sqrt(N) / np.sqrt(2 * np.pi)) * np.log(1.0 / epsilon)
        actual_iters.append(it)

        theoretical_iters.append(it_theory)

        print(f"{N:<5} | {it:<15} | {it_theory:<15.2f}")

    plt.figure(figsize=(10, 6))
    plt.plot(N_values, actual_iters, "o-", label="Experimantal", linewidth=2)
    plt.plot(
        N_values,
        theoretical_iters,
        "--",
        label=f"Theoretical function {method_form[method]}",
        color="red",
    )

    plt.title(f'"Method: {methods[method]} | ($\epsilon={epsilon}$)')
    plt.xlabel("Grid size (N)")
    plt.ylabel("Number of iterations (i)")
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.show()


N_values = [10, 20, 30, 40, 50, 60, 80]

draw_iters(N_values, 1e-6, 1)